In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

data_path = Path("../data/processed/chennai_house_price_prediction_cleaned.csv")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (7109, 19)


,PRT_ID,AREA,INT_SQFT,DIST_MAINROAD,N_BEDROOM,N_BATHROOM,N_ROOM,SALE_COND,PARK_FACIL,BUILDTYPE,UTILITY_AVAIL,STREET,MZZONE,QS_ROOMS,QS_BATHROOM,QS_BEDROOM,QS_OVERALL,COMMIS,SALES_PRICE
0,P03210,Karapakkam,1004,131,1.0,1.0,3,AbNormal,Yes,Commercial,AllPub,Paved,A,4.0,3.9,4.9,4.330,144400,7600000
1,P09411,Anna Nagar,1986,26,2.0,1.0,5,AbNormal,No,Commercial,AllPub,Gravel,RH,4.9,4.2,2.5,3.765,304049,21717770
2,P01812,Adyar,909,70,1.0,1.0,3,AbNormal,Yes,Commercial,ELO,Gravel,RL,4.1,3.8,2.2,3.090,92114,13159200
3,P05346,Velachery,1855,14,3.0,2.0,5,Family,No,Others,NoSewr,Paved,I,4.7,3.9,3.6,4.010,77042,9630290
4,P06210,Karapakkam,1226,84,1.0,1.0,3,AbNormal,Yes,Others,AllPub,Gravel,C,3.0,2.5,4.1,3.290,74063,7406250


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7109 entries, 0 to 7108
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   PRT_ID         7109 non-null   str    
 1   AREA           7109 non-null   str    
 2   INT_SQFT       7109 non-null   int64  
 3   DIST_MAINROAD  7109 non-null   int64  
 4   N_BEDROOM      7108 non-null   float64
 5   N_BATHROOM     7104 non-null   float64
 6   N_ROOM         7109 non-null   int64  
 7   SALE_COND      7109 non-null   str    
 8   PARK_FACIL     7109 non-null   str    
 9   BUILDTYPE      7109 non-null   str    
 10  UTILITY_AVAIL  7109 non-null   str    
 11  STREET         7109 non-null   str    
 12  MZZONE         7109 non-null   str    
 13  QS_ROOMS       7109 non-null   float64
 14  QS_BATHROOM    7109 non-null   float64
 15  QS_BEDROOM     7109 non-null   float64
 16  QS_OVERALL     7061 non-null   float64
 17  COMMIS         7109 non-null   int64  
 18  SALES_PRICE    7109

In [4]:
feature_summary = pd.DataFrame({
    "Data_Type": df.dtypes,
    "Unique_Values": df.nunique(),
    "Missing_Values": df.isnull().sum()
})

feature_summary

,Data_Type,Unique_Values,Missing_Values
PRT_ID,str,7109,0
AREA,str,7,0
INT_SQFT,int64,1699,0
DIST_MAINROAD,int64,201,0
N_BEDROOM,float64,4,1
N_BATHROOM,float64,2,5
N_ROOM,int64,5,0
SALE_COND,str,5,0
PARK_FACIL,str,2,0
BUILDTYPE,str,4,0


In [5]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (
        df.isnull().sum() / len(df)
    ) * 100
})

missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values(
    by="Missing_Count",
    ascending=False
)

missing_summary

,Missing_Count,Missing_Percentage
QS_OVERALL,48,0.675200
N_BATHROOM,5,0.070333
N_BEDROOM,1,0.014067


In [6]:
# Examine the affected rows
missing_rows = df[
    df[["N_BEDROOM", "N_BATHROOM", "QS_OVERALL"]]
    .isnull()
    .any(axis=1)
]

missing_rows

,PRT_ID,AREA,INT_SQFT,DIST_MAINROAD,N_BEDROOM,N_BATHROOM,N_ROOM,SALE_COND,PARK_FACIL,BUILDTYPE,UTILITY_AVAIL,STREET,MZZONE,QS_ROOMS,QS_BATHROOM,QS_BEDROOM,QS_OVERALL,COMMIS,SALES_PRICE
15,P06328,Velachery,1868,148,3.0,2.0,5,Family,No,Commercial,NoSewr,Gravel,RH,3.8,3.8,3.1,NaN,188569,15714080
46,P09483,T Nagar,1882,189,2.0,1.0,5,AbNormal,Yes,Others,ELO,Gravel,RM,3.9,4.5,4.7,NaN,272877,16051580
70,P05304,Anna Nagar,1589,39,1.0,NaN,4,Partial,No,Others,AllPub,Gravel,RL,2.1,2.5,4.5,3.160,85486,12212350
86,P00545,Chrompet,1085,149,1.0,1.0,3,AbNormal,Yes,Commercial,NoSewr,Gravel,RL,2.8,3.8,4.6,NaN,100545,14363500
111,P06768,Adyar,1034,73,1.0,1.0,3,AdjLand,Yes,Others,ELO,Gravel,C,3.8,3.3,3.5,NaN,46284,9256740
134,P00089,T Nagar,1942,10,2.0,1.0,5,Family,No,House,NoSeWa,Gravel,RH,4.6,3.1,2.9,NaN,61654,12330740
143,P06226,Karapakkam,1430,17,2.0,2.0,4,AbNormal,Yes,Others,NoSewr,No Access,A,4.3,3.2,3.8,NaN,106172,6635750
161,P06964,Velachery,1824,187,3.0,2.0,5,Normal Sale,No,House,NoSewr,Paved,I,3.9,2.3,4.1,NaN,76942,8549060
186,P08752,Adyar,738,69,1.0,1.0,2,Partial,No,Commercial,NoSewr,No Access,I,2.4,4.7,5.0,NaN,8313,8312670
208,P03292,Karapakkam,1578,153,2.0,2.0,4,Normal Sale,Yes,Others,AllPub,Gravel,I,4.3,4.6,3.4,NaN,238313,9532500


In [7]:
df["N_BEDROOM"].value_counts(dropna=False).sort_index()

N_BEDROOM
1.0    3795
2.0    2352
3.0     707
4.0     254
NaN       1
Name: count, dtype: int64

In [8]:
df.loc[
    df["N_BEDROOM"].isnull(),
    ["AREA", "INT_SQFT", "N_ROOM", "N_BATHROOM", "SALES_PRICE"]
]

,AREA,INT_SQFT,N_ROOM,N_BATHROOM,SALES_PRICE
4060,Anna Nagar,1556,4,1.0,10797790


In [9]:
df["N_BATHROOM"].value_counts(dropna=False).sort_index()

N_BATHROOM
1.0    5589
2.0    1515
NaN       5
Name: count, dtype: int64

In [10]:
df.loc[
    df["N_BATHROOM"].isnull(),
    ["AREA", "INT_SQFT", "N_BEDROOM", "N_ROOM", "SALES_PRICE"]
]

,AREA,INT_SQFT,N_BEDROOM,N_ROOM,SALES_PRICE
70,Anna Nagar,1589,1.0,4,12212350
5087,Chrompet,1016,1.0,3,10647920
6134,Chrompet,916,1.0,3,8847420
6371,Chrompet,1035,1.0,3,8274200
6535,Anna Nagar,1864,2.0,5,15556920


In [11]:
df["N_BEDROOM"].value_counts(dropna=False).sort_index()

N_BEDROOM
1.0    3795
2.0    2352
3.0     707
4.0     254
NaN       1
Name: count, dtype: int64

In [12]:
df["QS_OVERALL"].describe()

count    7061.000000
mean        3.503254
std         0.527223
min         2.000000
25%         3.130000
50%         3.500000
75%         3.890000
max         4.970000
Name: QS_OVERALL, dtype: float64

In [13]:
df.loc[
    df["QS_OVERALL"].isnull(),
    ["AREA", "INT_SQFT", "BUILDTYPE", "SALES_PRICE"]
]

,AREA,INT_SQFT,BUILDTYPE,SALES_PRICE
15,Velachery,1868,Commercial,15714080
46,T Nagar,1882,Others,16051580
86,Chrompet,1085,Commercial,14363500
111,Adyar,1034,Others,9256740
134,T Nagar,1942,House,12330740
143,Karapakkam,1430,Others,6635750
161,Velachery,1824,House,8549060
186,Adyar,738,Commercial,8312670
208,Karapakkam,1578,Others,9532500
248,Velachery,1776,House,9989170


In [14]:
df.groupby("BUILDTYPE", dropna=False).agg(
    Total_Properties=("QS_OVERALL", "size"),
    Missing_Quality_Scores=(
        "QS_OVERALL",
        lambda s: s.isnull().sum()
    )
)

,Total_Properties,Missing_Quality_Scores
BUILDTYPE,,
Commercial,2329,18
House,2444,10
Other,26,1
Others,2310,19


In [15]:
# Implement the Missing-Value Treatment
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Define the imputers
bedroom_imputer = SimpleImputer(strategy="median")

bathroom_imputer = SimpleImputer(strategy="most_frequent")

quality_imputer = SimpleImputer(strategy="median")

In [16]:
df.isnull().sum().sort_values(ascending=False)

QS_OVERALL       48
N_BATHROOM        5
N_BEDROOM         1
INT_SQFT          0
AREA              0
PRT_ID            0
DIST_MAINROAD     0
SALE_COND         0
PARK_FACIL        0
BUILDTYPE         0
N_ROOM            0
UTILITY_AVAIL     0
STREET            0
QS_ROOMS          0
MZZONE            0
QS_BATHROOM       0
QS_BEDROOM        0
COMMIS            0
SALES_PRICE       0
dtype: int64

In [17]:
# Step 1: Create the imputer
bedroom_imputer = SimpleImputer(strategy="median")

# Step 2: Learn the median and fill missing values
df["N_BEDROOM"] = bedroom_imputer.fit_transform(
    df[["N_BEDROOM"]]
).ravel()

In [18]:
# Fill missing bathroom counts
df["N_BATHROOM"] = bathroom_imputer.fit_transform(
    df[["N_BATHROOM"]]
).ravel()

# Fill missing overall quality scores
df["QS_OVERALL"] = quality_imputer.fit_transform(
    df[["QS_OVERALL"]]
).ravel()

In [19]:
df[["N_BEDROOM", "N_BATHROOM", "QS_OVERALL"]].isnull().sum()

N_BEDROOM     0
N_BATHROOM    0
QS_OVERALL    0
dtype: int64

In [22]:
output_path = Path('../data/processed/chennai_house_price_prediction_cleaned.csv')

# Create the output directory if it does not exist
output_path.parent.mkdir(parents=True, exist_ok=True)

# Save the cleaned data
df.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully!")
print("Location:", output_path)
print("Final shape:", df.shape)

Cleaned dataset saved successfully!
Location: ..\data\processed\chennai_house_price_prediction_cleaned.csv
Final shape: (7109, 19)
